In [ ]:
# Cell 1 — Install dependencies
!pip install unsloth trl transformers datasets torch matplotlib
!pip install "openenv-core>=0.2.3"
!pip install fastapi uvicorn pydantic requests openai pytest

In [ ]:
# Cell 2 — Clone repo and set working directory
import os, sys

REPO_URL = "https://huggingface.co/spaces/akkiisfrommars/ilyushin"

if not os.path.exists("ilyushin"):
    !git clone {REPO_URL} ilyushin

os.chdir("ilyushin")
sys.path.insert(0, ".")
print("Working directory:", os.getcwd())

In [ ]:
# Cell 3 — Set environment variables
import os
from google.colab import userdata

os.environ["HF_TOKEN"]     = userdata.get("HF_TOKEN")
os.environ["ENV_BASE_URL"] = "ws://localhost:8000"
os.environ["MODEL_NAME"]   = "Qwen/Qwen2.5-1.5B-Instruct"

print("HF_TOKEN set    :", bool(os.environ.get("HF_TOKEN")))
print("ENV_BASE_URL    :", os.environ["ENV_BASE_URL"])
print("MODEL_NAME      :", os.environ["MODEL_NAME"])

In [ ]:
# Cell 4 — Start the Ilyushin environment server
import subprocess
import time
import requests

server_process = subprocess.Popen(
    ["python", "run.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("Waiting for server to start...")
time.sleep(6)

try:
    res = requests.get("http://localhost:8000/")
    print("Server running:", res.json())
except Exception as e:
    print("Server error:", e)
    print(server_process.stderr.read().decode())

In [ ]:
# Cell 5 — Verify environment works via OpenEnv WebSocket client
from openenv.core.generic_client import GenericEnvClient

with GenericEnvClient(base_url="ws://localhost:8000").sync() as env:
    result = env.reset(task_id="easy")
    print("Reset successful")

    result = env.step({"type": "read_logs", "target_service": None})
    print("Step successful")
    print("Done:", result.done if hasattr(result, "done") else "?")

print("Environment verified OK")

In [ ]:
# Cell 6 — Run BASELINE (random agent) to establish starting point
import json
import random
from training.reward_fn import IlyushinRewardFn

print("Running BASELINE (random agent) episodes...")
print("=" * 50)

baseline_scores = {}
actions_pool    = ["read_logs", "restart_service", "check_metrics", "scale_up"]
services_pool   = ["web_server", "database", "cache", "queue", "api_gateway"]

for task_id in ["easy", "medium", "hard"]:
    scores = []
    for episode in range(10):
        env_fn = IlyushinRewardFn(task_id=task_id)
        try:
            state = env_fn.reset()
            done  = False
            steps = 0
            while not done and steps < 20:
                action = {
                    "type": random.choice(actions_pool),
                    "target_service": random.choice(services_pool)
                }
                state, reward, done = env_fn.step(json.dumps(action))
                steps += 1
            score = state.get("healthy_services", 0) / max(state.get("total_services", 5), 1)
            scores.append(score)
        finally:
            env_fn.close()
    avg = sum(scores) / len(scores)
    baseline_scores[task_id] = avg
    print(f"  {task_id:<10} avg_score = {avg:.2f}")

overall_baseline = sum(baseline_scores.values()) / len(baseline_scores)
print("=" * 50)
print(f"  Overall baseline score: {overall_baseline:.2f}")
print("\nBaseline established. Starting training...")

In [ ]:
# Cell 7 — Run GRPO training with curriculum learning
from training.train import train
train()

In [ ]:
# Cell 8 — Plot training reward curves
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

try:
    reward_img = mpimg.imread("training/plots/final_reward_curves.png")
    axes[0].imshow(reward_img)
    axes[0].axis("off")
    axes[0].set_title("Episode Reward Curves", fontsize=14)
except Exception:
    axes[0].text(0.5, 0.5, "No reward plot yet", ha="center")

try:
    score_img = mpimg.imread("training/plots/final_score_curves.png")
    axes[1].imshow(score_img)
    axes[1].axis("off")
    axes[1].set_title("Agent Score Across Phases", fontsize=14)
except Exception:
    axes[1].text(0.5, 0.5, "No score plot yet", ha="center")

plt.suptitle("Ilyushin — GRPO Training Results", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("training/plots/colab_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training/plots/colab_summary.png")

In [ ]:
# Cell 9 — Evaluate TRAINED agent and compare against baseline
from unsloth import FastLanguageModel
from training.train import run_episode
from training.reward_fn import IlyushinRewardFn

print("Loading trained model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    "training/checkpoints/final_model",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model loaded.")

print("\nEvaluating TRAINED agent...")
print("=" * 50)

trained_scores = {}
for task_id in ["easy", "medium", "hard"]:
    scores = []
    for _ in range(5):
        env_fn = IlyushinRewardFn(task_id=task_id)
        try:
            reward, steps, score, actions = run_episode(model, tokenizer, task_id, env_fn)
            scores.append(score)
        finally:
            env_fn.close()
    avg = sum(scores) / len(scores)
    trained_scores[task_id] = avg
    print(f"  {task_id:<10} avg_score = {avg:.2f}")

print("=" * 50)
print("\nCOMPARISON")
print(f"{'TASK':<12} {'BASELINE':<12} {'TRAINED':<12} {'IMPROVEMENT'}")
print("-" * 50)
for task_id in ["easy", "medium", "hard"]:
    b = baseline_scores.get(task_id, 0)
    t = trained_scores.get(task_id, 0)
    diff = t - b
    print(f"  {task_id:<10} {b:<12.2f} {t:<12.2f} {diff:+.2f}")

In [ ]:
# Cell 10 — Plot baseline vs trained comparison bar chart
import matplotlib.pyplot as plt
import numpy as np

tasks  = ["easy", "medium", "hard"]
x      = np.arange(len(tasks))
width  = 0.35

b_vals = [baseline_scores.get(t, 0) for t in tasks]
t_vals = [trained_scores.get(t, 0)  for t in tasks]

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, b_vals, width, label="Baseline (random)",  color="#e74c3c", alpha=0.8)
bars2 = ax.bar(x + width/2, t_vals, width, label="Trained (GRPO)",     color="#2ecc71", alpha=0.8)

ax.set_xlabel("Task Difficulty", fontsize=13)
ax.set_ylabel("Average Score (healthy services / total)", fontsize=13)
ax.set_title("Ilyushin — Baseline vs Trained Agent", fontsize=15, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([t.upper() for t in tasks], fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=12)
ax.axhline(y=1.0, color="black", linewidth=1, linestyle="--", alpha=0.4, label="Perfect")
ax.grid(True, alpha=0.3, axis="y")

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig("training/plots/baseline_vs_trained.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training/plots/baseline_vs_trained.png")